In [ ]:
from matplotlib import pyplot as plt

import numpy as np
from scipy.integrate import solve_ivp
from types import SimpleNamespace as sn
from tqdm import trange
from copy import deepcopy
from datetime import datetime
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Button, Slider

import viewer as Viewer
import simulator as Simulator
TestSim = Simulator.TestSimulation
View = Viewer.View

In [ ]:
import os

isdir = os.path.isdir('tests/')
path='tests/'


if os.path.isdir(path):
    sub_files = os.scandir(path)
for f in sub_files:
    if ('.npz' in f.name):
        print(f)

<built-in method __format__ of posix.DirEntry object at 0x124f1f320>
<built-in method __format__ of posix.DirEntry object at 0x124ed6bf0>
<built-in method __format__ of posix.DirEntry object at 0x124f1f320>
<built-in method __format__ of posix.DirEntry object at 0x124ed6bf0>
<built-in method __format__ of posix.DirEntry object at 0x124f1f320>
<built-in method __format__ of posix.DirEntry object at 0x124ed6bf0>
<built-in method __format__ of posix.DirEntry object at 0x124f1f320>
<built-in method __format__ of posix.DirEntry object at 0x124ed6bf0>
<built-in method __format__ of posix.DirEntry object at 0x124f1f320>


# Random Positions Test

In [ ]:

path = 'tests/random_positions/default_params.ini'
path = 'configs/small_test.ini'

data = TestSim(path)
time_steps = data['time_steps']
boid_count = data['boid_count']

NUM_REPLICAS=20
tests = []

for r in range(NUM_REPLICAS):
    data = TestSim(path)
    tests.append(data)

plt.ion()
View(tests,True)

NameError: name 'matplotlib' is not defined

# Concepts

## Ensamble Average
Essentially the expected value for a stochastic thingymabob

## Determinent
(2D) The transformation matrix for a vector such that area is 0. Think about squiching the two $\hat{i}$ and $\hat{j}$ onto the same line

## Eigenvectors and Eigenvalues
The line on which a given vector stays on its span after transformation; doesn't rotate, but does scale.<br>
An Eigenvector $\vec{v}$ is any vector for which above is true (given some transformation matrix $A$), and the amount the vector is scaled, is the Eigenvalue $\lambda$
### $A\vec{v}=\lambda\vec{v}$
where $A$ is the transformation matrix, $\vec{v}$ is the eigenvector, and $\lambda$ is the eigen value<br>
equation states that the matrix-vector multiplication $A\vec{v}$ gives the same as a simple scalar multiplication $\lambda\vec{v}$.
<br>
Working with the equation above isn't intuitive because the left side represents a matrix multiplication, and the right side, a scalar multiplication.
<br> Thus, representing $\lambda$ as a matrix (for which only scaling by lambda occurs) makes it simpler:<br>
$\begin{bmatrix}\lambda & 0 & 0\\0 & \lambda & 0\\0 & 0 & \lambda\end{bmatrix} = \lambda\begin{bmatrix}1 & 0 & 0\\0 & 1 & 0\\0 & 0 & 1\end{bmatrix}$
<br>
### $A\vec{v}=(\lambda I)\vec{v}$
### $A\vec{v}-(\lambda I)\vec{v} = \vec{0}$
### $\vec{v}(A-\lambda I) = \vec{0}$

$\vec{v}(A-\lambda I) \rightarrow \begin{bmatrix} a-\lambda & b & c\\d & e-\lambda & f\\g & h & i-\lambda\end{bmatrix}$
<br>$\therefore$ We're looking for a vector ($\vec{v}$) which when multiplied with above gives the 0 vector.
### $\vec{v} \begin{bmatrix} a-\lambda & b & c\\d & e-\lambda & f\\g & h & i-\lambda\end{bmatrix} = \vec{0}$
<br>

if we want the product of a non 0 vector $\vec{v}$ and a matrix to equal 0, we need the determinent of the matrix to be 0. Or in other words, the matrix needs to squish the space into a lower dimension.

[Video visualising this effect](https://youtu.be/PFDu9oVAE-g?t=457)

## Eigendecomposition
https://www.youtube.com/watch?v=KTKAp9Q3yWg
Useful for when you need to multiply a matrix by itself. If a matrix has some eigenvector(s), we can simplify process

## Replica Definition
A replica in this case is some theoretical arbitrary run of the simulation, which shares it's input signal with the set of replicas, but has different starting conditions otherwise.
<br>
### $x(t) = s(t) + n(t)$
where $x(t)$ is the response signal of a replica at time $t$, in other words, the state of the simulation; the boid positions and velocities,<br>
where $n(t)$ is the noise, the unexpected/chaotic portion of the signal<br>
where $s(t)$ is the signal component, the expected portion of the signal (see more in next section).

## Signal Component
Specifically, the signal component $s(t)$ is the *ensamble average* state of a replica at time $t$. If we imagine it as discrete opposed to continuos, it's the largest bin if we were to binomially distribute the possible states at $t$.

### Finding the ensamble average
The state of the reservoir for a given time $t$ and number of boids $N$, has two componenets, its velocity and position: $v(t),p(t) \in R^{ND}$<br>
We can concatinate these two into a single state space $x(t) = \begin{bmatrix} p(t)\\ v(t)\end{bmatrix} \in R^{ND}$
<br>
Thus, finding the ensamble average: 
#### $\langle x(t) \rangle = \frac{1}{R} \sum\limits_{r}^{R} x_r(t)\\ \therefore \\s(t) = \frac{1}{R} \sum\limits_{r}^{R} x_r(t)$

where R is the total number of replicas.

In [4]:
path = 'configs/small_test.ini'
data = TestSim(path)
time_steps = data['time_steps']
boid_count = data['boid_count']

NUM_REPLICAS=20

x=np.concatenate((data['positions'],data['velocities']),1)
x_shape = [NUM_REPLICAS]
x_shape.extend(list(x.shape))
x_r = np.empty(x_shape)
x_r.shape


100%|██████████| 9/9 [00:00<00:00, 87.59it/s]


(20, 10, 400, 2)

In [35]:
for r in range(NUM_REPLICAS):
    data = TestSim(path)
    x=np.concatenate((data['positions'],data['velocities']),1)
    x_r[r]=x

x_mean = x_r.mean(0)


100%|██████████| 9/9 [00:00<00:00, 98.39it/s]


# Reservoir Profiling/Consistency Analysis
These section creates test for profiling 

# Consistency Profile


## Finding the signal
where the ensamble **signal** $s$ is some average state of the reservoir at a given time $t$.<br>
Where we do many different simulations with different starting conditions (replicas), and by "averaging" the state of each replica at a given time $t$,<br>
we gain an understanding of what the *default* state is for the reservoir at a given $t$, given the same inputs (same lorenz)<br>




In [ ]:
NUM_REPLICAS = 10
TIME_STEPS = 100

sim = Simulator.run_simulation
load_params = Simulator.load_ini

## Consistency Correlation
### $\gamma^2 = \frac{\langle s^2 \rangle}{\langle (s+n)^2\rangle}$
(where the angled brackets are the inner product of presumably two vectors)<br>
Each replica may be decomposed into a signal and a noise component<br>where typically $i=2$, as to say there are two responses (Lymburn makes this assertion) <br>
$x_i(t) = s_i(t) + n_i(t)$<br>
$x_i(t) = s_i(t) + n_i(t)$<br>
The signal component $s(t)$ is defined as the ensemble average of an infinite number of replicas,<br>
and the noise $n(t)$ component is defined as the remainder for each replica.<br>
<br>
x is the reservoir state $\therefore$ positions and vectors of the swarm.
<br>

# Observation Layer